1(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-AraElectra-cross-encoder-KD-v1)
2(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-AraElectra-cross-encoder-NoKD-v1)
3(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-KD-v1)
4(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-NoKD-v1)
5(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-mMiniLML-cross-encoder-KD-v1)
6(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-mMiniLML-cross-encoder-NoKD-v1)
7(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-AraElectra-bi-encoder-KD-v1)
8(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-AraElectra-bi-encoder-NoKD-v1)
9(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-KD-v1)
10(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-AraDPR-bi-encoder-NoKD-v1)
11(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-mMiniLML-bi-encoder-KD-v1)
12(https://huggingface.co/hatemestinbejaia/mmarco-Arabic-mMiniLML-bi-encoder-NoKD-v1)

In [ ]:
from huggingface_hub import hf_hub_download
organization_dataset_id="hatemestinbejaia/STCALIR_Synthetic-Test-Collection"
hf_hub_download(repo_id=organization_dataset_id, filename="Corpus Algerian Legal Texts.csv", local_dir="./", repo_type="dataset")
import pandas
df = pandas.read_csv("Corpus Algerian Legal Texts.csv")
df = df[['passage_id', 'text_with_summary']]
# Assume your DataFrame is df_passages
df = df.rename(columns={'passage_id': 'id', 'text_with_summary': 'text'})
df["id"] = df["id"].replace(' ','_', regex=True)
# Optional: reset the index
df = df.reset_index(drop=True)
df.to_csv('STCALIR_collection.tsv', sep="\t", header=None,index=False) 
df.head(1)

In [ ]:
from huggingface_hub import hf_hub_download
organization_dataset_id="hatemestinbejaia/STCALIR_Synthetic-Test-Collection"
hf_hub_download(repo_id=organization_dataset_id, filename="Topics Algerian Legal Texts.csv", local_dir="./", repo_type="dataset")
df_topic = pandas.read_csv("Topics Algerian Legal Texts.csv")
df_topic = df_topic.reset_index()[["topic_id", "topic_title"]]
df_topic = df_topic.rename(columns={'topic_id': 'id', 'topic_title': 'text'})
df_topic.to_csv('STCALIR-Topics.tsv', sep="\t", header=None,index=False) 
print(df_topic.head())

In [ ]:
hf_hub_download(repo_id=organization_dataset_id, filename="STCALIR_bm25_6faiss_rrf.txt", local_dir="./", repo_type="dataset")

In [ ]:
!head STCALIR_bm25_6faiss_rrf.txt

In [ ]:
!pip install sentence_transformers -q
#!pip install huggingface_hub -q

In [ ]:
organization_dataset_id="hatemestinbejaia/ExperimentDATA_knowledge_distillation_vs_fine_tuning"
#qerl_file ="qrels.msmarco-passage.dev-subset.txt"
collection_name= "STCALIR_collection.tsv"
queries_name= "STCALIR-Topics.tsv"
list_of_models= [ "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
              "hatemestinbejaia/mmarco-Arabic-AraElectra-cross-encoder-NoKD-v1",
              "hatemestinbejaia/mmarco-Arabic-AraElectra-cross-encoder-KD-v1",
              "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-NoKD-v1",
              "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-KD-v1",
              "hatemestinbejaia/mmarco-Arabic-mMiniLML-cross-encoder-NoKD-v1",
              "hatemestinbejaia/mmarco-Arabic-mMiniLML-cross-encoder-KD-v1",
            ]
# reranked 
ranked_list=["STCALIR_bm25_6faiss_rrf.txt"]

In [ ]:
import pandas as pd
#defaut number of condidat document K is 1000
def load_run(path, k=1000):
    print('Loading run...')
    run = pd.read_csv(path, delim_whitespace=True, header=None) # remove 
    #print(run.head(10))
    run = run[run[2] <= k] 
    #print(run.head(10))
    run = run.groupby(0)[1].apply(list).to_dict()
    return run

In [ ]:
import pandas as pd
import tqdm
queries = {}
for i in tqdm.trange(len(df_topic)):
    queries[df_topic['id'][i]]= df_topic['text'][i]
    #break

In [ ]:
import pandas as pd
corpus = {}
for i in tqdm.trange(len(df)):
    corpus[df['id'][i]]= df['text'][i]
    #break

In [ ]:
import operator
import torch
def reranking_crossEncoder(query_id, passage_list_id, threshold):
    corpora=[]
    query = queries[query_id]
    X=[]
    for j in passage_list_id : 
        corpora.append([query, corpus[j]])
        X.append(j)
    with torch.no_grad():scores = model.predict(corpora, batch_size=8, show_progress_bar=False)
    #print(scores)
    x = dict(zip(X, scores))
    #print(x)
    xr = dict(sorted(x.items(), key=operator.itemgetter(1), reverse=True))
    # Filter dictionary to keep only scores > threshold
    #xr_filtered = {doc: score for doc, score in xr.items() if score > threshold}
    #xr_filtered_sorted = dict(sorted(xr_filtered.items(), key=lambda x: x[1], reverse=True))

    #print("***************************************")
    #print("***************************************")
    #return list(xr.keys())
    return list(xr.items())

In [ ]:
from sentence_transformers import CrossEncoder, SentenceTransformer, util
import time
from tqdm import tqdm
for i in list_of_models :
    model = CrossEncoder(i, max_length=256)
    run = load_run(ranked_list[0], 1000)
    output_run= i.split("/")[1]+"_pseudo_judgment.tsv"
    print(output_run)
    #start = time.time()
    for k in tqdm(run.keys()):
        run[k] = reranking_crossEncoder(k, run[k], 0.9)
        #print(type(run[k]))    
        #break
        #end = time.time()
        #print(end - start)
    # Run reranker
    with open(output_run, 'w', encoding='utf-8') as f_out:
        for k in run.keys():
            rank=1
            for j in run[k]:
                f_out.write(f'{k}\t{j[0]}\t{rank}\t{j[1]}\n')
                rank =rank +1
                
    print("Done!")
    #break

In [ ]:
import os
prefix = "_pseudo_judgment.tsv"
files = [f for f in os.listdir(".") if f.endswith(prefix)]
print(files)

In [ ]:
from huggingface_hub import login
access_token = "your hugginface apikey"
# This saves your token for use by Transformers, pipelines, etc.
login(token=access_token)

In [ ]:
organization_dataset_id="hatemestinbejaia/STCALIR_Synthetic-Test-Collection"
from huggingface_hub import HfApi
api = HfApi()
for i in files:
    api.upload_file(
    path_or_fileobj=i,
    path_in_repo=i,
    repo_id=organization_dataset_id,
    repo_type="dataset",
    )

In [ ]:
import pandas as pd
from collections import defaultdict
import glob

def load_ranking_file(path):
    """Load a ranking file of format: qid, docid, rank."""
    return pd.read_csv(path, sep="\t", names=["qid", "docid", "rank", "score"])

def rrf(ranking_lists, k=60):
    """Apply RRF fusion to a list of ranking lists (docid lists)."""
    scores = defaultdict(float)
    for ranking in ranking_lists:
        for rank, docid in enumerate(ranking):
            scores[docid] += 1.0 / (k + rank + 1)
    # Sort documents by RRF score
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def rrf_fusion(run_files, output_path=None, k=60, top_k=1000):
    """
    Apply RRF to multiple ranking files with identical structure.
    
    run_files: list of file paths
    output_path: where to save fused run (optional)
    Returns: fused DataFrame
    """
    
    # 1) Load all runs
    runs = [load_ranking_file(f) for f in run_files]

    # All topics from first file
    topics = runs[0]["qid"].unique()
    #topics = sorted(set().union(*[df["qid"].unique() for df in runs]))


    fused_rows = []

    for qid in topics:
        ranking_lists = []

        # 2) Extract docids per topic from each run
        for df in runs:
            docs = df[df["qid"] == qid].sort_values("rank")["docid"].tolist()
            ranking_lists.append(docs)

        # 3) Apply RRF
        fused_docs = rrf(ranking_lists, k=k)

        # 4) Keep top_k
        for rank, (docid, score) in enumerate(fused_docs[:top_k], start=1):
            fused_rows.append([qid, docid, rank])

    # Create DataFrame
    fused_df = pd.DataFrame(fused_rows, columns=["qid", "docid", "rank"])

    # Optional save
    if output_path:
        fused_df.to_csv(output_path, sep="\t", index=False, header=False)
        print(f"Saved RRF run to: {output_path}")

    return fused_df
rrf_fusion(files, output_path="STCALIR_7cross_encoder_rrf.txt")

In [ ]:
api.upload_file(
    path_or_fileobj="STCALIR_7cross_encoder_rrf.txt",
    path_in_repo="STCALIR_7cross_encoder_rrf.txt",
    repo_id=organization_dataset_id,
    repo_type="dataset",
    )